In [53]:
from pyscf import gto, scf, cc

a = 2 # bond length in a cluster
d = 100 # distance between each cluster
unit = 'b' # unit of length
na = 2 # size of a cluster (monomer)
nc = 3 # set as integer multiple of monomers
spin = 0 # spin per monomer
frozen = 0 # frozen orbital per monomer
elmt = 'H'
basis = 'sto6g'
# for nc in nc_list:
atoms = ""
for n in range(nc*na):
    shift = ((n - n % na) // na) * (d-a)
    atoms += f"{elmt} {n*a+shift:.5f} 0.00000 0.00000 \n"

mol = gto.M(atom=atoms, basis="sto6g", unit='B', spin=0, verbose=4)
mol.build()

mf = scf.RHF(mol)
mf.kernel()

mo = mf.stability()[0]
dm = mf.make_rdm1(mo,mf.mo_occ)
mf.kernel(dm0=dm)
mo = mf.stability()[0]
dm = mf.make_rdm1(mo,mf.mo_occ)
mf.kernel(dm0=dm)

nfrozen = 0
mycc = cc.CCSD(mf,frozen=nfrozen)
mycc.kernel()[0]

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-19-generic', version='#19~24.04.2-Ubuntu SMP PREEMPT_DYNAMIC Fri Mar  6 23:08:46 UTC 2', machine='x86_64')  Threads 16
Python 3.11.14 (main, Oct 21 2025, 18:31:21) [GCC 11.2.0]
numpy 2.3.1  scipy 1.16.2  h5py 3.14.0
Date: Thu Mar 26 15:49:14 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 6
[INPUT] num. electrons = 6
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = B
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 H      0.000000000000   0.000000000000   0.000000000000 AA    

np.float64(-0.11892420274654766)

In [54]:
# example for PT2

options = {'n_eql': 3,
           'n_prop_steps': 50,
            'n_ene_blocks': 1,
            'n_sr_blocks': 5,
            'n_blocks': 50,
            'n_walkers': 300,
            'seed': 17,
            'walker_type': 'rhf',
            'trial': 'stoccsd',
            'nslater': 10,
            'dt':0.005,
            'use_gpu': False,
            }

from ad_afqmc.prop_unrestricted.mixed_wave import prep
import jax
jax.config.update("jax_enable_x64", True)
prep.prep_afqmc(mycc,chol_cut=1e-5)
# prop_unrestricted.run_afqmc(options,nproc=1)
option_file='options.bin'
import pickle
with open(option_file, 'wb') as f:
    pickle.dump(options, f)

#
# Preparing AFQMC calculation
# Calculating Cholesky integrals
# Finished calculating Cholesky integrals
#
# Size of the correlation space:
# Number of electrons: (3, 3)
# Number of basis functions: 6
# Number of Cholesky vectors: 9
#


In [55]:
import time
import numpy as np
from jax import random
from jax import numpy as jnp
from functools import partial 

ham_data, ham, prop, trial, wave_data, sampler, options = (prep._prep_afqmc())

init_time = time.time()

### initialize propagation
init_walkers = None
trial_rdm1 = trial.get_rdm1(wave_data)
if "rdm1" not in wave_data:
    wave_data["rdm1"] = trial_rdm1
ham_data = ham.build_measurement_intermediates(ham_data, trial, wave_data)
ham_data = ham.build_propagation_intermediates(ham_data, prop, trial, wave_data)

prop_data = prop.init_prop_data(trial, wave_data, ham_data, init_walkers)
if jnp.abs(jnp.sum(prop_data["overlaps"])) < 1.0e-6:
    raise ValueError(
        "Initial overlaps are zero. Pass walkers with non-zero overlap."
    )
prop_data["key"] = random.PRNGKey(options["seed"])

# wave_data['tau'], _ = trial.decompose_t2(wave_data)

prop_data["overlaps"] = trial.calc_overlap(prop_data["walkers"], wave_data)
prop_data["n_killed_walkers"] = 0

e_init= jnp.real(trial.calc_energy(prop_data["walkers"], ham_data, wave_data)[0])
# ept_sp = h0 + e0/t1 + e1/t1 - t2 * e0 / t1**2
# ept = jnp.array(jnp.sum(ept_sp) / prop.n_walkers)
prop_data["e_estimate"] = e_init
prop_data["pop_control_ene_shift"] = prop_data["e_estimate"]

print(e_init)
print(e_init-mf.e_tot)

# Throw 6 vectors in T2 deomposition
# cutoff = 1.00e-08 | error = 1.09e-16
# number of T2 decomposition vectors 3
# nelec: (3, 3)
# norb: 6
# nchol: 9
# n_eql: 3
# n_prop_steps: 50
# n_ene_blocks: 1
# n_sr_blocks: 5
# n_blocks: 50
# n_walkers: 300
# seed: 17
# walker_type: rhf
# trial: stoccsd
# nslater: 10
# dt: 0.005
# use_gpu: False
# n_exp_terms: 6
# n_batch: 1
# max_error: 0.001
-3.169289646225913
8.881784197001252e-16


In [21]:
xtaus, prop_data = trial.get_xtaus(prop_data, wave_data, prop)
olp_cc, energy_cc = trial.calc_energy_stoccsd(prop_data["walkers"], xtaus, ham_data, wave_data)

In [26]:
olp = prop_data["overlaps"]
wt = prop_data["weights"]
estocc = jnp.sum(wt * olp_cc/olp * energy_cc)  / jnp.sum(wt * olp_cc/olp)
print(estocc)
print(mycc.e_tot)

(-1.0941962724683456-1.5023738762374772e-17j)
-1.0960712830549568


In [30]:
nsample = 1000
weights = np.zeros(nsample)
stocc_os = np.zeros(nsample, dtype="complex128")
stocc_es = np.zeros(nsample, dtype="complex128")
for n in range(nsample):
    xtaus, prop_data = trial.get_xtaus(prop_data, wave_data, prop)
    olp_cc, ene_cc = trial.calc_energy_stoccsd(prop_data["walkers"], xtaus, ham_data, wave_data)
    wt_walkers, olp_walkers = prop_data["weights"], prop_data["overlaps"]
    
    weight = jnp.sum(wt_walkers)
    olp_cc_walkers = jnp.sum(wt_walkers * olp_cc / olp_walkers) / weight
    ene_cc_walkers = jnp.sum(wt_walkers * olp_cc / olp_walkers * ene_cc) / weight
    weights[n] = weight
    stocc_os[n] = olp_cc_walkers
    stocc_es[n] = ene_cc_walkers

    stocc_den = jnp.sum(weights[:n+1] * stocc_os[:n+1]) / jnp.sum(weights[:n+1])
    stocc_num = jnp.sum(weights[:n+1] * stocc_es[:n+1]) / jnp.sum(weights[:n+1])
    stocc_energy = jnp.real(stocc_num / stocc_den)

    print(f'{n+1}  {weight.real:.6f}  {olp_cc_walkers.real:.6f}  {ene_cc_walkers.real:.6f}  {stocc_energy:.6f}')

1  300.000000  10.000000  -10.971202  -1.097120
2  300.000000  10.000000  -10.951318  -1.096126
3  300.000000  10.000000  -10.962965  -1.096183
4  300.000000  10.000000  -10.967732  -1.096330
5  300.000000  10.000000  -10.965597  -1.096376
6  300.000000  10.000000  -10.968813  -1.096460
7  300.000000  10.000000  -10.979745  -1.096677
8  300.000000  10.000000  -10.946111  -1.096419
9  300.000000  10.000000  -10.950825  -1.096270
10  300.000000  10.000000  -10.943575  -1.096079
11  300.000000  10.000000  -10.954176  -1.096019
12  300.000000  10.000000  -10.958796  -1.096007
13  300.000000  10.000000  -10.942637  -1.095873
14  300.000000  10.000000  -10.965891  -1.095924
15  300.000000  10.000000  -10.945019  -1.095829
16  300.000000  10.000000  -10.972971  -1.095921
17  300.000000  10.000000  -10.952469  -1.095881
18  300.000000  10.000000  -10.954816  -1.095859
19  300.000000  10.000000  -10.955551  -1.095843
20  300.000000  10.000000  -10.967526  -1.095889
21  300.000000  10.000000  -1

In [31]:
mycc.e_tot

np.float64(-1.0960712830549568)

In [38]:
key = jax.random.PRNGKey(0)
key, subkey = jax.random.split(key)
rand_walkers = jax.random.normal(subkey, shape=prop_data["walkers"].shape)
xtaus, prop_data = trial.get_xtaus(prop_data, wave_data, prop)
olp_cc, energy_cc = trial.calc_energy_stoccsd(rand_walkers, xtaus, ham_data, wave_data)
olp = trial.calc_overlap(rand_walkers, wave_data)
wt = prop_data["weights"]
estocc = jnp.sum(wt * olp_cc/olp * energy_cc)  / jnp.sum(wt * olp_cc/olp)
print(estocc)
print(mycc.e_tot)

(-0.946771986422709-0.003410962919144208j)
-1.0960712830549568


In [39]:
nsample = 1000
weights = np.zeros(nsample)
stocc_os = np.zeros(nsample, dtype="complex128")
stocc_es = np.zeros(nsample, dtype="complex128")
for n in range(nsample):
    xtaus, prop_data = trial.get_xtaus(prop_data, wave_data, prop)
    olp_cc, ene_cc = trial.calc_energy_stoccsd(rand_walkers, xtaus, ham_data, wave_data)
    # olp_walkers = trial.calc_overlap(rand_walkers, wave_data)
    # wt_walkers = prop_data["weights"]
    # wt_walkers, olp_walkers = prop_data["weights"], prop_data["overlaps"]
    
    # weight = jnp.sum(wt_walkers)
    olp_cc_walkers = jnp.sum(olp_cc)
    ene_cc_walkers = jnp.sum(olp_cc * ene_cc)
    # weights[n] = weight
    stocc_os[n] = olp_cc_walkers
    stocc_es[n] = ene_cc_walkers

    stocc_den = jnp.sum(stocc_os[:n+1]) # / jnp.sum(weights[:n+1])
    stocc_num = jnp.sum(stocc_es[:n+1]) # / jnp.sum(weights[:n+1])
    stocc_energy = jnp.real(stocc_num / stocc_den)

    print(f'{n+1}  {weight.real:.6f}  {olp_cc_walkers.real:.6f}  {ene_cc_walkers.real:.6f}  {stocc_energy:.6f}')

1  300.000000  2383.981268  -2559.814761  -1.073755
2  300.000000  2341.934539  -2554.736244  -1.081982
3  300.000000  2379.948278  -2562.765443  -1.080359
4  300.000000  2289.590214  -2553.873377  -1.088774
5  300.000000  2308.062867  -2551.549059  -1.092130
6  300.000000  2309.170855  -2552.947445  -1.094257
7  300.000000  2352.238796  -2559.563961  -1.093488
8  300.000000  2356.670678  -2557.500407  -1.092429
9  300.000000  2335.030513  -2554.386008  -1.092629
10  300.000000  2303.693875  -2554.888140  -1.094249
11  300.000000  2312.604665  -2557.272920  -1.095289
12  300.000000  2315.947741  -2554.729248  -1.095934
13  300.000000  2279.324374  -2565.926173  -1.098180
14  300.000000  2294.273989  -2556.725772  -1.099323
15  300.000000  2301.601148  -2554.036734  -1.100003
16  300.000000  2265.095267  -2555.260762  -1.101717
17  300.000000  2342.605310  -2563.345093  -1.101269
18  300.000000  2329.121395  -2558.457782  -1.101109
19  300.000000  2368.200866  -2555.422513  -1.099936
20

In [ ]:
from jax import jit, lax

@partial(jit, static_argnums=(0,3,4))
def _block(self, prop_data, ham_data, prop, trial, wave_data):
    """Block scan function. Propagation and energy calculation."""

    prop_data["key"], subkey = random.split(prop_data["key"])
    fields = random.normal(
        subkey,
        shape=(
            self.n_prop_steps,
            prop.n_walkers,
            self.n_chol,
        ),
    )
    _step_scan_wrapper = lambda x, y: self._step_scan(
        x, y, ham_data, prop, trial, wave_data
    )
    prop_data, _ = lax.scan(_step_scan_wrapper, prop_data, fields)
    prop_data["n_killed_walkers"] += prop_data["weights"].size - jnp.count_nonzero(
        prop_data["weights"]
    )

    # raondom fields_x for T2 decomposition
    xtaus, prop_data = trial.get_xtaus(prop_data, wave_data, prop)
    prop_data = prop.orthonormalize_walkers(prop_data)
    # prop_data = prop.stochastic_reconfiguration_local(prop_data)

    olp_hf = trial.calc_overlap(prop_data["walkers"], wave_data)
    prop_data["overlaps"] = olp_hf
    ene_hf = jnp.real(trial.calc_energy(prop_data["walkers"], ham_data, wave_data))
    olp_cc, ene_cc = trial.calc_energy_stoccsd(prop_data["walkers"], xtaus, ham_data, wave_data)
    wt_hf = prop_data["weights"]

    weight = jnp.sum(wt_hf)
    ehf = jnp.sum(wt_hf * ene_hf) / weight
    num = jnp.sum(wt_hf * olp_cc / olp_hf * ene_cc) / weight
    den = jnp.sum(wt_hf * olp_cc / olp_hf) / weight

    prop_data = prop.stochastic_reconfiguration_local(prop_data)
    prop_data["overlaps"] = trial.calc_overlap(prop_data["walkers"], wave_data)

    return prop_data, (weight, ehf, num, den)


@partial(jit, static_argnums=(0,3,4))
def _block_froze(self, prop_data, ham_data, prop, trial, wave_data):
    """Block scan function. Propagation and energy calculation."""

    prop_data["key"], subkey = random.split(prop_data["key"])
    # fields = random.normal(
    #     subkey,
    #     shape=(
    #         self.n_prop_steps,
    #         prop.n_walkers,
    #         self.n_chol,
    #     ),
    # )
    # _step_scan_wrapper = lambda x, y: self._step_scan(
    #     x, y, ham_data, prop, trial, wave_data
    # )
    # prop_data, _ = lax.scan(_step_scan_wrapper, prop_data, fields)
    # prop_data["n_killed_walkers"] += prop_data["weights"].size - jnp.count_nonzero(
    #     prop_data["weights"]
    # )

    # raondom fields_x for T2 decomposition
    xtaus, prop_data = trial.get_xtaus(prop_data, wave_data, prop)
    # prop_data = prop.orthonormalize_walkers(prop_data)
    # prop_data = prop.stochastic_reconfiguration_local(prop_data)

    olp_hf = trial.calc_overlap(prop_data["walkers"], wave_data)
    prop_data["overlaps"] = olp_hf
    ene_hf = jnp.real(trial.calc_energy(prop_data["walkers"], ham_data, wave_data))
    olp_cc, ene_cc = trial.calc_energy_stoccsd(prop_data["walkers"], xtaus, ham_data, wave_data)
    wt_hf = prop_data["weights"]

    weight = jnp.sum(wt_hf)
    ehf = jnp.sum(wt_hf * ene_hf) / weight
    num = jnp.sum(wt_hf * olp_cc / olp_hf * ene_cc) / weight
    den = jnp.sum(wt_hf * olp_cc / olp_hf) / weight

    # prop_data = prop.stochastic_reconfiguration_local(prop_data)
    # prop_data["overlaps"] = trial.calc_overlap(prop_data["walkers"], wave_data)

    return prop_data, (weight, ehf, num, den)

In [ ]:
trial.nslater = 10
xtaus, prop_data = trial.get_xtaus(prop_data, wave_data, prop)
olp_cc, ene_cc = trial.calc_energy_stoccsd(prop_data["walkers"], xtaus, ham_data, wave_data)
olp_hf = prop_data["overlaps"]
wt_hf = prop_data["weights"]
ecc_init = jnp.real(jnp.sum(wt_hf*olp_cc/olp_hf*ene_cc)/jnp.sum(wt_hf*olp_cc/olp_hf))
print(ecc_init)
print(mycc.e_tot)

-3.2876359857271695
-3.2882138489724615


In [57]:
from ad_afqmc.prop_unrestricted.mixed_wave import sampling
ehf_init = jnp.real(prop_data["e_estimate"])

den, num = trial.calc_energy_stoccsd(prop_data["walkers"], xtaus, ham_data, wave_data)
ecc_init = jnp.real(jnp.sum(num) / jnp.sum(den))

print(f'# Propagating with {options["n_walkers"]} walkers')
print("# Equilibration sweeps:")
print("# atom_time    ehf    e_stocc    Walltime")
print(f"  {0.:.2f}  {ehf_init:.6f}   {ecc_init:.6f}  {time.time() - init_time:.2f}")

sampler_eq = sampling.sampler_stoccsd(
    n_prop_steps=50, 
    n_ene_blocks=1, 
    n_sr_blocks=1, 
    n_chol = sampler.n_chol
    )

block_time = prop.dt * sampler_eq.n_prop_steps * sampler_eq.n_ene_blocks * sampler_eq.n_sr_blocks

for n in range(100):
    prop_data, (whf, ehf, num, den) \
        = _block(sampler_eq, prop_data, ham_data, prop, trial, wave_data)

    ecc = jnp.real(num / den)

    prop_data["e_estimate"] = 0.9 * prop_data["e_estimate"] + 0.1 * ecc

    print(f" {(n+1)*block_time:.2f}  {whf:.6f}  {ehf:.6f}  {num:.6f}  {den:.6f}  {ecc:.6f}  {time.time() - init_time:.2f} ")


# Propagating with 300 walkers
# Equilibration sweeps:
# atom_time    ehf    e_stocc    Walltime
  0.00  -3.169290   -0.328764  29.49
 0.25  300.300783  -3.195714  -33.660584+0.008509j  10.237478-0.003011j  -3.287976  31.29 
 0.50  300.456089  -3.217255  -34.804227+0.002669j  10.614001-0.000528j  -3.279086  33.02 
 0.75  300.530059  -3.233018  -35.082927+0.023072j  10.675192-0.007032j  -3.286398  33.08 
 1.00  300.677278  -3.249418  -35.225303-0.049043j  10.695733+0.013213j  -3.293398  33.14 
 1.25  300.746012  -3.256696  -35.522986+0.029188j  10.775085-0.014845j  -3.296768  33.21 
 1.50  300.397014  -3.258254  -35.526314+0.110308j  10.777648-0.041188j  -3.296287  33.27 
 1.75  300.499496  -3.266002  -36.095516-0.066086j  10.978304+0.024274j  -3.287893  33.33 
 2.00  300.437093  -3.264398  -36.126687+0.006264j  10.987724-0.002851j  -3.287913  33.39 
 2.25  300.270090  -3.267811  -36.027739+0.081443j  10.953791-0.026983j  -3.289064  33.45 
 2.50  300.291350  -3.274153  -36.476162+0.0092

In [ ]:
print("# Sampling sweeps:")
print("# nBlocks  energy/hf  error  energy/cc  error  Walltime")
sampler.n_blocks = 2000

whf_sp = np.zeros(sampler.n_blocks,dtype="float64")
ehf_sp = np.zeros(sampler.n_blocks,dtype="float64")
num_sp = np.zeros(sampler.n_blocks,dtype="complex128")
den_sp = np.zeros(sampler.n_blocks,dtype="complex128")

for n in range(sampler.n_blocks):
    prop_data, (whf, ehf, num, den) \
        = _block_froze(sampler_eq, prop_data, ham_data, prop, trial, wave_data)

    whf_sp[n] = whf
    ehf_sp[n] = ehf
    num_sp[n] = num
    den_sp[n] = den
    ecc_estimate = jnp.real(num / den)

    prop_data["e_estimate"] = 0.9 * prop_data["e_estimate"] + 0.1 * ecc_estimate

    # if (n+1) % (max(sampler.n_blocks // 10, 1)) == 0 and n > 0:
    if n > 0:
        whf = np.sum(whf_sp[:n+1]) 
        whf_ehf = np.sum(whf_sp[:n+1] * ehf_sp[:n+1])
        whf_num = np.sum(whf_sp[:n+1] * num_sp[:n+1])
        whf_den = np.sum(whf_sp[:n+1] * den_sp[:n+1])

        ehf = np.real(whf_ehf / whf)
        ehf_err = np.sqrt(np.sum(whf_sp[:n+1] * (ehf_sp[:n+1]-ehf)**2 / whf)) / np.sqrt(n)

        ecc = np.real(whf_num / whf_den)

        ecc_sp = (whf_sp[:n+1] * num_sp[:n+1]) / (whf_sp[:n+1] * den_sp[:n+1])
        ecc_sp_mean = ecc_sp.mean()
        ecc_sp_mean_std = np.real(np.std(ecc_sp) / np.sqrt(len(ecc_sp)))

        print(f" {n+1:4d}  {ehf:.6f}  {ehf_err:.6f}  {ecc_estimate:.6f}  {ecc:.6f}  {ecc_sp_mean_std:.6f}  {time.time() - init_time:.2f}")

# Sampling sweeps:
# nBlocks  energy/hf  error  energy/cc  error  Walltime
    2  -3.268801  0.000000  -3.302916  -3.292797  0.007116  46.12
    3  -3.268801  0.000000  -3.299790  -3.295121  0.005099  46.13
    4  -3.268801  0.000000  -3.282642  -3.291986  0.004695  46.14
    5  -3.268801  0.000000  -3.294656  -3.292521  0.003840  46.14
    6  -3.268801  0.000000  -3.281385  -3.290641  0.003638  46.15
    7  -3.268801  0.000000  -3.292855  -3.290958  0.003131  46.16
    8  -3.268801  0.000000  -3.286614  -3.290412  0.002788  46.16
    9  -3.268801  0.000000  -3.307105  -3.292240  0.003036  46.17
   10  -3.268801  0.000000  -3.289914  -3.292008  0.002758  46.17
   11  -3.268801  0.000000  -3.287955  -3.291639  0.002536  46.18
   12  -3.268801  0.000000  -3.282857  -3.290901  0.002430  46.19
   13  -3.268801  0.000000  -3.291567  -3.290952  0.002248  46.19
   14  -3.268801  0.000000  -3.291242  -3.290973  0.002091  46.20
   15  -3.268801  0.000000  -3.272217  -3.289695  0.002298  46.21
 

In [ ]:
print("# Sampling sweeps:")
print("# nBlocks  energy/hf  error  energy/cc  error  Walltime")
sampler.n_blocks = 2000

whf_sp = np.zeros(sampler.n_blocks,dtype="float64")
ehf_sp = np.zeros(sampler.n_blocks,dtype="float64")
num_sp = np.zeros(sampler.n_blocks,dtype="complex128")
den_sp = np.zeros(sampler.n_blocks,dtype="complex128")

for n in range(sampler.n_blocks):
    prop_data, (whf, ehf, num, den) \
        = _block(sampler_eq, prop_data, ham_data, prop, trial, wave_data)

    whf_sp[n] = whf
    ehf_sp[n] = ehf
    num_sp[n] = num
    den_sp[n] = den
    ecc_estimate = jnp.real(num / den)

    prop_data["e_estimate"] = 0.9 * prop_data["e_estimate"] + 0.1 * ecc_estimate

    # if (n+1) % (max(sampler.n_blocks // 10, 1)) == 0 and n > 0:
    if n > 0:
        whf = np.sum(whf_sp[:n+1])
        whf_ehf = np.sum(whf_sp[:n+1] * ehf_sp[:n+1])
        whf_num = np.sum(whf_sp[:n+1] * num_sp[:n+1])
        whf_den = np.sum(whf_sp[:n+1] * den_sp[:n+1])

        ehf = np.real(whf_ehf / whf)
        ehf_err = np.sqrt(np.sum(whf_sp[:n+1] * (ehf_sp[:n+1]-ehf)**2 / whf)) / np.sqrt(n)

        ecc = np.real(whf_num / whf_den)
        ecc_sp = (whf_sp[:n+1] * num_sp[:n+1]) / (whf_sp[:n+1] * den_sp[:n+1])
        ecc_err = np.real(np.std(ecc_sp) / np.sqrt(len(ecc_sp)))

        print(f" {n+1:4d}  {ehf:.6f}  {ehf_err:.6f}  {ecc_estimate:.6f}  {ecc:.6f}  {ecc_sp_mean_std:.6f}  {time.time() - init_time:.2f}")

# Sampling sweeps:
# nBlocks  energy/hf  error  energy/cc  error  Walltime
    2  -3.285489  0.002603  -3.279029  -3.289717  0.007677  66.25
    3  -3.291105  0.005811  -3.313889  -3.297710  0.008310  66.31
    4  -3.287130  0.005719  -3.303626  -3.299152  0.006373  66.37
    5  -3.282957  0.006088  -3.301003  -3.299515  0.005108  66.43
    6  -3.280554  0.005523  -3.277799  -3.295868  0.005401  66.49
    7  -3.281357  0.004736  -3.308801  -3.297684  0.004931  66.55
    8  -3.282504  0.004258  -3.286375  -3.296236  0.004522  66.61
    9  -3.282657  0.003759  -3.295801  -3.296188  0.004020  66.68
   10  -3.283651  0.003506  -3.301702  -3.296738  0.003668  66.74
   11  -3.282584  0.003346  -3.302924  -3.297291  0.003376  66.80
   12  -3.281718  0.003175  -3.295995  -3.297184  0.003097  66.86
   13  -3.281347  0.002944  -3.293487  -3.296901  0.002873  66.92
   14  -3.280190  0.002961  -3.290599  -3.296455  0.002704  66.98
   15  -3.279868  0.002775  -3.298401  -3.296584  0.002527  67.04
 

In [60]:
mycc.e_tot

np.float64(-3.2882138489724615)